# 1. IMPORTS

In [1]:
import re, ast, csv
from pathlib import Path
from typing import Any, Dict
import numpy as np
import soundfile as sf
import torch
import librosa

# 2. RUTAS

In [2]:
REPO_DIR  = Path(r"C:\TFG\aasist") #ruta al repo AASIST
CONF_PATH = REPO_DIR / "config" / "AASIST.conf" #ruta al archivo de configuración
CKPT_PATH = REPO_DIR / "models" / "weights" / "AASIST.pth" #ruta al checkpoint del modelo  

INPUT_DIR = Path(r"C:\TFG\Datasets\Propio") #ruta a los audios
OUT_CSV   = Path(r"C:\TFG\tfg-2526-xplicavoz\\Modelos\German\aasist_results.csv") #ruta al csv de salida

print("CONF existe:", CONF_PATH.exists(), CONF_PATH)
print("CKPT existe:", CKPT_PATH.exists(), CKPT_PATH)

CONF existe: True C:\TFG\aasist\config\AASIST.conf
CKPT existe: True C:\TFG\aasist\models\weights\AASIST.pth


# 3. LEER CONF
El archivo .conf contiene arquitectura del modelo, parámetros de entrenamiento, dataset y su protocolo, preprocesado y checkpoints.

La función busca el elemento pasado por parámetro y devuelve el bloque superior que contiene ese elemento junto con los elementos al mismo nivel.

In [4]:
txt = CONF_PATH.read_text(encoding="utf-8", errors="ignore")

def extract_enclosing_dict(text: str, elem: str) -> str:
    i = text.find(elem)
    if i == -1:
        raise RuntimeError(f"No encuentro el elemento a buscar {elem!r} en el conf.")
    j = i
    while j >= 0 and text[j] != "{":
        j -= 1
    if j < 0:
        raise RuntimeError("No encuentro '{' antes del elem.")

    depth = 0
    in_str = False
    esc = False
    k = j
    while k < len(text):
        ch = text[k]
        if in_str:
            if esc:
                esc = False
            elif ch == "\\":
                esc = True
            elif ch == '"':
                in_str = False
        else:
            if ch == '"':
                in_str = True
            elif ch == "{":
                
                depth += 1
            elif ch == "}":
                depth -= 1
                if depth == 0:
                    return text[j:k+1]
        k += 1
    raise RuntimeError("Error leyendo conf.")

# 4. CONSTUIR d_args
d_args contiene:
- first_conv: parámetros del primer bloque convolucional del modelo
- filts: estructura que marca cuántos filtros usa el modelo en cada bloque.
- gat_dims: dimensiones internas del embedding que representa cada nodo.
- pool_ratios: lista de ratios usados para pooling en las distintas fases.
- temperatures: controla lo suave o agresiva que es la atención en cada fase.

In [5]:
block = extract_enclosing_dict(txt, '"first_conv"')
block = re.sub(r",\s*(\}|\])", r"\1", block)  # quita comas colgantes
d_args: Dict[str, Any] = ast.literal_eval(block)

required = ["filts", "gat_dims", "pool_ratios", "temperatures", "first_conv"]
missing = [k for k in required if k not in d_args]
print("d_args listo. Características sin cargar:", missing)
if missing:
    raise RuntimeError("No se extrajo el dict correcto del conf.")

d_args listo. Características sin cargar: []


# 5. INSTANCIAR EL MODELO

In [6]:
from AASIST import Model

device = "cpu"
model = Model(d_args).to(device).eval()
print("Modelo instanciado")

Modelo instanciado


# 6. CARGAR CHEKPOINT
El checkpoint sirve para cargar los pesos del entrenamiento

In [7]:
state = torch.load(str(CKPT_PATH), map_location=device)
if isinstance(state, dict):
    for key in ("state_dict", "model", "model_state_dict", "net"):
        if key in state and isinstance(state[key], dict):
            state = state[key]
            break

if isinstance(state, dict) and any(k.startswith("module.") for k in state.keys()):
    state = {k.replace("module.", "", 1): v for k, v in state.items()}

missing_keys, unexpected_keys = model.load_state_dict(state, strict=False)
print(f"Checkpoint cargado (missing={len(missing_keys)}, unexpected={len(unexpected_keys)})")
r = model.eval()

Checkpoint cargado (missing=0, unexpected=0)


# 6.2. GUARDAR MODELO PARA EL PIPELINE

In [8]:
torch.save(model, "aasist_full.pt")
print("Guardado en aasist_full.pt")

Guardado en aasist_full.pt


# 7. HELPERS DE AUDIO + VENTANA DESLIZANTE
Los helpers sirven para dejar los audios en un modelo estándar: mono(un canal), 16 Khz.
La ventana sirve para audios más largos de 4 segundos, los parte en ventanas de 4 segundos con solapamiento de dos para que no se quede información en el borde de una ventana.

In [8]:
def load_mono_16k(path, target_sr=16000):
    x, sr = sf.read(str(path), always_2d=False)
    x = x.astype(np.float32)
    if x.ndim > 1:
        x = np.mean(x, axis=1).astype(np.float32)
    if sr != target_sr:
        x = librosa.resample(x, orig_sr=sr, target_sr=target_sr).astype(np.float32)
    x = np.clip(x, -1.0, 1.0).astype(np.float32)
    return x


def make_windows(x: np.ndarray, sr=16000, win_sec=4.0, hop_sec=2.0):
    win = int(win_sec * sr)
    hop = int(hop_sec * sr)
    if len(x) <= win:
        y = np.zeros(win, dtype=np.float32)
        y[:len(x)] = x
        return [y]
    out=[]
    for s in range(0, len(x)-win+1, hop):
        out.append(x[s:s+win])
    if (len(x)-win) % hop != 0:
        out.append(x[-win:])
    return out

def pad(x, max_len=64600):
    x_len = x.shape[0]
    if x_len >= max_len:
        return x[:max_len]

    num_repeats = int(max_len / x_len) + 1
    padded_x = np.tile(x, (1, num_repeats))[:, :max_len][0]
    return padded_x

# 8. INFERENCIA

In [9]:
#Este hace media de probabilidades de todas las ventanas
@torch.no_grad()
def average_windows(wav: Path, win_sec=4.0, hop_sec=2.0):
    x = load_mono_16k(wav, 16000)
    ws = make_windows(x, 16000, win_sec, hop_sec)
    ps=[]
    for w in ws:
        t = torch.from_numpy(w).unsqueeze(0).to(device)  # (1,T)

        out = model(t)
        if isinstance(out, (tuple, list)):
            logits = next(o for o in out if torch.is_tensor(o) and o.ndim == 2 and o.shape[1] == 2)
        else:
            logits = out


        p1 = torch.softmax(logits, dim=1)[0,1].item()
        ps.append(p1)

    return float(np.mean(ps)), len(x)/16000.0, len(ws) #devuelve la probabilidad media contando todas las ventanas

In [10]:
#Este sería sin ventanas 
CUT = 64600  # ~4.04s 

@torch.no_grad()
def score_without_windows(wav: Path):
    x = load_mono_16k(wav, 16000)              
    x4 = pad(x, CUT).astype(np.float32)  #ajusta el audio a la longitud que espera el modelo   

    t = torch.from_numpy(x4).unsqueeze(0).to(device) #convierte el audio a tensor 

    out = model(t)
    if isinstance(out, (tuple, list)):
        logits = next(o for o in out if torch.is_tensor(o) and o.ndim == 2 and o.shape[1] == 2)
    else:
        logits = out

    score_bonafide = logits[0, 1].item()

    prob_bonafide = torch.softmax(logits, dim=1)[0, 1].item()

    duration_sec = len(x) / 16000.0
    return score_bonafide, prob_bonafide, duration_sec


In [11]:
#Este sumando los logists de todas las ventanas antes de hacer la media
CUT = 64600

@torch.no_grad()
def average_logits(wav: Path, win_sec=4.0, hop_sec=2.0):
    x = load_mono_16k(wav, 16000)
    ws = make_windows(x, 16000, win_sec, hop_sec)

    logits_sum = None

    for w in ws:
        w = pad(w, CUT).astype(np.float32)
        t = torch.from_numpy(w).unsqueeze(0).to(device)

        _, logits = model(t) 

        logits_sum = logits if logits_sum is None else logits_sum + logits

    prob_bona = torch.softmax(logits_sum, dim=1)[0, 1].item()
    score_bona = logits_sum[0, 1].item() 

    return prob_bona, score_bona, len(x)/16000.0, len(ws)


# 9. TRATAR LOS AUDIOS

In [12]:
files = sorted(INPUT_DIR.rglob("*.wav"))
print("Audios encontrados:", len(files))
if not files:
    raise RuntimeError("No hay .wav en la ruta.")

with OUT_CSV.open("w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["filename", "prob1_mean", "duration_sec", "n_windows"])
    for i, fp in enumerate(files, 1):
        #p1, dur, nw = average_windows(fp, win_sec=4.0, hop_sec=2.0)
        #prob_bona, score_bona, dur, nw = average_logits(fp, win_sec=4.0, hop_sec=2.0)
        #w.writerow([str(fp), prob_bona, dur, nw])
        score_bona, prob_bona, dur = score_without_windows(fp)
        w.writerow([str(fp), prob_bona, dur])

print("Resultados en:", OUT_CSV)

Audios encontrados: 200
Resultados en: C:\TFG\tfg-2526-xplicavoz\Modelos\German\aasist_results.csv
